# Étape 2 — MLFlow tracking

Je mets en place le tracking MLFlow (backend SQLite + artifacts locaux) et je lance deux baselines : Logistic Regression et Random Forest. Le but est juste de valider que les runs s'enregistrent bien et qu'on peut les comparer dans l'UI. La comparaison de modèles sérieuse arrive en Étape 3.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import importlib, training
importlib.reload(training)

tracking_uri = training.setup_mlflow()
print("Tracking URI:", tracking_uri)
print("Expérience  :", training.EXPERIMENT_NAME)

## Chargement

In [ ]:
X, y = training.load_training_data()
print("X:", X.shape, "  y mean:", round(y.mean(), 4))

## Baseline 1 — Logistic Regression

Pipeline imputation médiane + scaling + LR pondérée. 3 folds stratifiés pour rester rapide à l'Étape 2 (5 folds en Étape 3).

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

logreg = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
    ("clf", LogisticRegression(
        class_weight="balanced", max_iter=1000, solver="lbfgs", random_state=42,
    )),
])

metrics_lr = training.cv_run(
    logreg, X, y,
    run_name="logreg_baseline",
    n_splits=3,
    extra_params={"model": "LogisticRegression", "class_weight": "balanced", "max_iter": 1000},
    extra_tags={"step": "2", "family": "linear"},
)
metrics_lr

## Baseline 2 — Random Forest

Bornes serrées (100 arbres, profondeur 10) pour que ça tienne quelques minutes.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("clf", RandomForestClassifier(
        n_estimators=100, max_depth=10,
        class_weight="balanced", n_jobs=-1, random_state=42,
    )),
])

metrics_rf = training.cv_run(
    rf, X, y,
    run_name="rf_baseline",
    n_splits=3,
    extra_params={"model": "RandomForest", "n_estimators": 100, "max_depth": 10, "class_weight": "balanced"},
    extra_tags={"step": "2", "family": "tree-ensemble"},
)
metrics_rf

## Vérification côté MLFlow

Les deux runs doivent apparaître dans l'expérience `credit-default`. L'UI se lance avec `.\.venv\Scripts\mlflow.exe ui --backend-store-uri sqlite:///mlruns.db --default-artifact-root file:./mlartifacts` depuis la racine du projet.

In [ ]:
import mlflow
from mlflow.tracking import MlflowClient

client = MlflowClient()
exp = client.get_experiment_by_name(training.EXPERIMENT_NAME)
runs = mlflow.search_runs(experiment_ids=[exp.experiment_id], order_by=["start_time DESC"])
cols = [c for c in ["tags.mlflow.runName", "metrics.auc_mean", "metrics.recall_minority_mean", "metrics.business_cost_mean", "start_time"] if c in runs.columns]
runs[cols]